# Load Libraries and packages

In [1]:
import pandas as pd
import csv
import os

In [ ]:
file_name = "082026_soy_yield_paper_states.csv" #file to get from extracted
save_name = 'df_soy_yield_2026.csv' #name to save new df under in created_dfs_step1

# Data Collection

In [3]:
# 1) Read the soy yield CSV into a pandas DataFrame
full_path = os.path.join("extracted_usda_soy_data", file_name)
df_corn_yield = pd.read_csv(full_path)

# 2) Select only the columns we care about:
df_corn_yield = df_corn_yield[
    ["Year", "State ANSI", "County ANSI", "Ag District Code", "Data Item", "Value", "CV (%)"]
]

#3) Rename the columns to your preferred naming:
df_corn_yield.columns = ["year", "state_ansi", "county_ansi", "district_code", "data_item", "value", "cv"]

#orint the corn yield DataFrame sample
print("Corn Yield DataFrame:\n", df_corn_yield.head())

#print all unique items in the Data Item column ;)
unique_data_items = df_corn_yield["data_item"].unique()
print("\nUnique Data Items in the corn yield data:")
print(unique_data_items)

Corn Yield DataFrame:
    year  state_ansi  county_ansi  district_code  \
0  2025          17        115.0             40   
1  2025          17        123.0             40   
2  2025          17        125.0             40   
3  2025          17        113.0             40   
4  2025          17        143.0             40   

                                 data_item  value   cv  
0  SOYBEANS - YIELD, MEASURED IN BU / ACRE   75.9  2.1  
1  SOYBEANS - YIELD, MEASURED IN BU / ACRE   73.5  3.7  
2  SOYBEANS - YIELD, MEASURED IN BU / ACRE   68.5  4.1  
3  SOYBEANS - YIELD, MEASURED IN BU / ACRE   74.0  1.2  
4  SOYBEANS - YIELD, MEASURED IN BU / ACRE   71.6  2.2  

Unique Data Items in the corn yield data:
['SOYBEANS - YIELD, MEASURED IN BU / ACRE']


In [4]:
#remove '99' districts
df_corn_yield = df_corn_yield[df_corn_yield["district_code"] != "99"]

#drop rows where county_ansi is NaN (prevents IntCastingNaNError)
df_corn_yield = df_corn_yield.dropna(subset=["county_ansi"])

#convert county_ansi to int
df_corn_yield["county_ansi"] = df_corn_yield["county_ansi"].astype(int)

#(Optional) Convert county_ansi back to a zero-padded string (3 digits) to match NOAA if needed
df_corn_yield["county_ansi"] = df_corn_yield["county_ansi"].astype(str).str.zfill(3)

#reverse district_code if you need "10" → "01" for matching NOAA divisions
df_corn_yield["district_code"] = df_corn_yield["district_code"].apply(
    lambda x: str(x)[::-1].zfill(2)
)

print(df_corn_yield[["year", "state_ansi", "county_ansi", "district_code"]].head())

   year  state_ansi county_ansi district_code
0  2025          17         115            04
1  2025          17         123            04
2  2025          17         125            04
3  2025          17         113            04
4  2025          17         143            04


In [ ]:
# Compute the min and max year for each state in the corn yield dataset
state_year_ranges = df_corn_yield.groupby("state_ansi")["year"].agg(["min", "max"])
print("Year ranges per state:")
print(state_year_ranges)

# Check-and-report only, no trimming: the real floor on how far back the
# weather-merged data can go is 1951 (nclimgrid-daily's start, enforced per-county
# by 03b's importer), so that's the only thing worth flagging here. Yield rows
# outside any state's own range simply never existed in the raw USDA export.
# (Nebraska/31 has no county-level soybean yield data before 1960 in the raw
# USDA export, checked 2026-08-23, so it will show up flagged here.)
WEATHER_FLOOR_YEAR = 1951
late_starting_states = state_year_ranges[state_year_ranges["min"] > WEATHER_FLOOR_YEAR]
if not late_starting_states.empty:
    print(f"\nFlag: states starting later than {WEATHER_FLOOR_YEAR} (the weather-data floor):")
    print(late_starting_states)
else:
    print(f"\nAll states have data at or before {WEATHER_FLOOR_YEAR}, none exceed the weather-data floor.")

In [7]:
# Only continue with the total yield data, not irrigated or non-irrigated
corn_yield_df = df_corn_yield[df_corn_yield["data_item"] == "SOYBEANS - YIELD, MEASURED IN BU / ACRE"]
print("Filtered Corn Yield DataFrame:")
print(corn_yield_df.head())

Filtered Corn Yield DataFrame:
   year  state_ansi county_ansi district_code  \
0  2025          17         115            04   
1  2025          17         123            04   
2  2025          17         125            04   
3  2025          17         113            04   
4  2025          17         143            04   

                                 data_item  value   cv  
0  SOYBEANS - YIELD, MEASURED IN BU / ACRE   75.9  2.1  
1  SOYBEANS - YIELD, MEASURED IN BU / ACRE   73.5  3.7  
2  SOYBEANS - YIELD, MEASURED IN BU / ACRE   68.5  4.1  
3  SOYBEANS - YIELD, MEASURED IN BU / ACRE   74.0  1.2  
4  SOYBEANS - YIELD, MEASURED IN BU / ACRE   71.6  2.2  


### Check for missing data

In [ ]:
# Ensure that the yield column is numeric and that 'year' is an integer
corn_yield_df["value"] = pd.to_numeric(corn_yield_df["value"], errors="coerce")
corn_yield_df["year"] = corn_yield_df["year"].astype(int)

# Dictionary to store the missing report (keyed by (state, county_ansi))
missing_report = {}

# Group by state and county to build the missing report
# Expected range is each state's own observed range (state_year_ranges, computed
# above), not one shared window forced across every state
for (state, county), group in corn_yield_df.groupby(["state_ansi", "county_ansi"]):
    # Get the years present in the group (even if the yield is NaN)
    years_present = set(group["year"].unique())
    # This state's own observed range
    state_min = state_year_ranges.loc[state, "min"]
    state_max = state_year_ranges.loc[state, "max"]
    # Compare against the dynamically computed expected range
    missing_rows = sorted(set(range(state_min, state_max + 1)) - years_present)
    
    # Find years where a row exists but the yield 'value' is NaN
    missing_values = sorted(group.loc[group["value"].isna(), "year"].unique())
    
    if missing_rows or missing_values:
        missing_report[(state, county)] = {
            "missing_rows": missing_rows,
            "missing_values": missing_values
        }

# Print the missing report header
print("Yield data covers each state's own observed year range (see state_year_ranges above)")
for (state, county), d in missing_report.items():
    print(f"State {state}, County {county}:")
    if d["missing_rows"]:
        print(f"  Rows missing for years: {d['missing_rows']}")
    if d["missing_values"]:
        print(f"  Years with missing yield value: {d['missing_values']}")

# Compute overall reporting counts
all_counties = list(corn_yield_df.groupby(["state_ansi", "county_ansi"]).groups.keys())
no_missing_count = len(all_counties) - len(missing_report)
exactly_one_missing = 0
more_than_one_missing = 0

for rep in missing_report.values():
    total_missing = len(rep["missing_rows"]) + len(rep["missing_values"])
    if total_missing == 1:
        exactly_one_missing += 1
    elif total_missing > 1:
        more_than_one_missing += 1

print("\nSummary:")
print(f"Total counties: {len(all_counties)}")
print(f"Counties with no missing data: {no_missing_count}")
print(f"Counties with exactly 1 missing year: {exactly_one_missing}")
print(f"Counties with more than 1 missing year: {more_than_one_missing}")

In [ ]:
# First, build the same missing_report as before …
# (I’ll assume you already have that dict around)

# Create a list of tuples: ((state, county), total_missing)
missing_counts = [
    ((state, county),
     len(d["missing_rows"]) + len(d["missing_values"]))
    for (state, county), d in missing_report.items()
]

# Sort descending by total_missing
missing_counts_sorted = sorted(
    missing_counts,
    key=lambda item: item[1],
    reverse=True
)

print("Counties sorted by total missing years (rows + values):")
for (state, county), total in missing_counts_sorted:
    d = missing_report[(state, county)]
    rows = d["missing_rows"]
    vals = d["missing_values"]
    print(f"State {state}, County {county}: {total} missing years")
    if rows:
        print(f"   • Missing rows: {rows}")
    if vals:
        print(f"   • Missing values: {vals}")

#### Save loaded and processed yield data

In [10]:
# Convert ANSI codes to strings with leading zeros:
#   state_ansi   → 2 digits (e.g. "04", "17")
#   county_ansi  → 3 digits (e.g. "001", "107")
#   district_code→ 2 digits

corn_yield_df = corn_yield_df.copy()

corn_yield_df['state_ansi']    = corn_yield_df['state_ansi'].astype(str).str.zfill(2)
corn_yield_df['county_ansi']   = corn_yield_df['county_ansi'].astype(str).str.zfill(3)
corn_yield_df['district_code'] = corn_yield_df['district_code'].astype(str).str.zfill(2)

# Save to CSV without the index
corn_yield_df.to_csv(os.path.join('created_dfs_step1', save_name), index=False)